<a href="https://colab.research.google.com/github/Gianbattistabsn/FAIML-RL-26/blob/main/part2/clone_colab.ipynb" target="_parent"> <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> </a>

## Setup
Clone the repo, build an isolated venv with the pinned RL stack, smoke-test imports, and log in to W&B.

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# ------------------------------------------------------------
# Paths / config — all subsequent cells reference these
# ------------------------------------------------------------
REPO_URL    = "https://github.com/Gianbattistabsn/FAIML-RL-26.git"
REPO_BRANCH = "main"
REPO_ROOT   = "/content/FAIML-RL-26"
VENV        = "/content/rl_env"
PYTHON      = f"{VENV}/bin/python"
PIP         = f"{VENV}/bin/pip"

# Mount Google Drive only if you want trained models to survive runtime recycling.
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

# Clone repo — re-clone if already present so each run picks up the latest pushes
if os.path.exists(REPO_ROOT):
    print(f"removing existing clone at {REPO_ROOT}")
    !rm -rf {REPO_ROOT}
!git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_ROOT}

In [ ]:
# ------------------------------------------------------------
# Build venv
# ------------------------------------------------------------
if not os.path.exists(VENV):

    !apt-get update -qq && apt-get install -y python3-venv

    !python -m venv --without-pip {VENV}
    
    !curl -sS https://bootstrap.pypa.io/get-pip.py | {PYTHON}
    
    !{PIP} install -U pip setuptools wheel
    
    !{PIP} install \
        numpy==1.26.4 \
        gymnasium==0.29.1 \
        stable-baselines3==2.3.2 \
        pybullet \
        tensorboard \
        pyvirtualdisplay \
        moviepy \
        imageio \
        imageio-ffmpeg \
        shimmy \
        opencv-python-headless==4.9.0.80 \
        tqdm rich wandb
    
    !{PIP} install -e {REPO_ROOT}/part2/panda-gym

    print(f"\nSETUP COMPLETE — venv at {VENV}")

else:
    print(f"venv already present at {VENV}, skipping install (delete it manually to rebuild)")

In [ ]:
# ------------------------------------------------------------
# Smoke test
# ------------------------------------------------------------
!{PYTHON} -c "import numpy as np, gymnasium, pybullet, panda_gym, torch; \
print('NumPy:    ', np.__version__); \
print('Gymnasium:', gymnasium.__version__); \
print('Torch:    ', torch.__version__); \
print('CUDA:     ', torch.cuda.is_available()); \
print('OK')"

In [ ]:
# wandb login 
!wandb login

## Training
Edit the config block, then run the cell. The strategy-specific flags (mass range, ADR/BADR knobs) are added to the command line only when used.

**Note:** `train_sb3.py` asks `use PPO? [y/n]` interactively at the beginning, and `render? [y/n]`at the end.

In [ ]:
# ------------------------------------------------------------
# Training config
# ------------------------------------------------------------
ALGO = "sac"             # "ppo" or "sac" — also used to derive MODEL_PATH in the eval section
ENV_TYPE = "source"      # "source" or "target"
SAMPLING = "none"        # "none" | "udr" | "adr" | "badr"
TIMESTEPS = 600_000
NO_VECNORM = True        # train without VecNormalize

# DR-specific (ignored when SAMPLING == "none")
MASS_MIN, MASS_MAX = 0.5, 2.0
ADR_DELTA = 0.2
ADR_BUFFER_SIZE = 20
ADR_PERF_LOW = -25
ADR_PERF_HIGH = -10
ADR_BOUNDARY_PROB = 0.8

# Build the command
extra_train_flags = ""
if SAMPLING in ("udr", "adr", "badr"):
    extra_train_flags += f"--mass-range {MASS_MIN} {MASS_MAX} "
if SAMPLING in ("adr", "badr"):
    extra_train_flags += (
        f"--adr-delta {ADR_DELTA} "
        f"--adr-buffer-size {ADR_BUFFER_SIZE} "
        f"--adr-perf-low {ADR_PERF_LOW} "
        f"--adr-perf-high {ADR_PERF_HIGH} "
        f"--adr-boundary-prob {ADR_BOUNDARY_PROB}"
    )
vecnorm_flag = "--no-vecnormalize" if NO_VECNORM else ""

TRAIN_SCRIPT = f"{REPO_ROOT}/part2/train_sb3.py"

In [ ]:
# ------------------------------------------------------------
# Training
# ------------------------------------------------------------
!MPLBACKEND=Agg {PYTHON} {TRAIN_SCRIPT} \
    --env-type {ENV_TYPE} \
    --sampling-strategy {SAMPLING} \
    --timesteps {TIMESTEPS} \
    {vecnorm_flag} {extra_train_flags}

## Evaluation
`MODEL_PATH`, `EVAL_USE_VECNORM` are derived from the training. Change manually when loading a model.

In [ ]:
# ------------------------------------------------------------
# Eval config - run for any kind of evaluation
# ------------------------------------------------------------
EVAL_SCRIPT = f"{REPO_ROOT}/part2/eval_sb3.py"
MASS_SWEEP_SCRIPT = f"{REPO_ROOT}/part2/eval_mass_sweep.py"
MODEL_PATH = f"/content/part2/models/{ALGO}_push_{SAMPLING}_{ENV_TYPE}_{TIMESTEPS // 1000}k.zip"
EVAL_EPISODES = 50  # requested by the teachers
EVAL_STOCHASTIC = False
EVAL_USE_VECNORM = not NO_VECNORM   # MUST match training-time normalization, or eval is meaningless

extra_eval_flags = ""
if not EVAL_USE_VECNORM:
    extra_eval_flags += "--no-vecnormalize "
if EVAL_STOCHASTIC:
    extra_eval_flags += "--stochastic"

print(f"model: {MODEL_PATH}")
print(f"vecnorm at eval: {EVAL_USE_VECNORM}, stochastic: {EVAL_STOCHASTIC}, episodes: {EVAL_EPISODES}")

In [ ]:
# Eval on the source env (baseline)
evaluate_source = True
if evaluate_source:
    !MPLBACKEND=Agg {PYTHON} {EVAL_SCRIPT} \
        --model-path {MODEL_PATH} \
        --env-type source \
        --episodes {EVAL_EPISODES} {extra_eval_flags}

In [ ]:
# Eval on the target env (sim2real)
evaluate_target = True
if evaluate_target:
    !MPLBACKEND=Agg {PYTHON} {EVAL_SCRIPT} \
        --model-path {MODEL_PATH} \
        --env-type target \
        --episodes {EVAL_EPISODES} {extra_eval_flags}

### Evaluation for different masses

In [ ]:
evaluate_sweep = True

masses = np.concatenate(([0.1], np.arange(0.5, 15.1, 0.5), [30, 50, 100]))

if evaluate_sweep:
    !MPLBACKEND=Agg {PYTHON} {MASS_SWEEP_SCRIPT} \
        --model-path {MODEL_PATH} \
        --env-type source \
        --masses {" ".join([f"{m:.1f}" for m in masses])} \
        --episodes {EVAL_EPISODES} {extra_eval_flags}

In [ ]:
# get saved statistics and plot
in_json = MODEL_PATH.replace(".zip", "_sweep_stats.json")
out_png_return = in_json.replace(".json", "_return.png")
out_png_success = in_json.replace(".json", "_success.png")

with open(in_json) as f:
    d = json.load(f)

# return vs mass
fig, ax = plt.subplots(figsize=(16, 9))
ax.errorbar(d["masses"], d["mean_returns"], yerr=d["std_returns"], fmt="o-", capsize=4)
ax.set_xlabel("mass (kg)")
ax.set_ylabel("mean return")
ax.set_title("return vs mass")
ax.grid(True, alpha=0.3)
fig.savefig(out_png_return, dpi=120, bbox_inches="tight")
fig.show()


# success rate vs mass — with mean-success-rate reference line
mean_sr = float(np.mean(d["success_rates"]))

fig, ax = plt.subplots(figsize=(16, 9))
ax.plot(d["masses"], d["success_rates"], "o-", label="per-mass success rate")
ax.axhline(mean_sr, ls="--", color="red",
           label=f"mean success rate = {mean_sr * 100:.1f}%")
ax.set_xlabel("mass (kg)")
ax.set_ylabel("success rate")
ax.set_title("success rate vs mass")
ax.grid(True, alpha=0.3)
ax.legend(loc="best")
fig.savefig(out_png_success, dpi=120, bbox_inches="tight")
fig.show()

print(f"return plot:  {out_png_return}")
print(f"success plot: {out_png_success}")